<a href="https://colab.research.google.com/github/mim-1999/ames-housing-regression/blob/main/Ames_Housing_EDA_LinearRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# import libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Create the 'images' directory if it doesn't exist
os.makedirs('images', exist_ok=True)

## Load dataset and understand it

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Ames_Housing_Project/train.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## check missing values

In [ ]:
df.isnull().sum()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
cols_with_missing = missing.index.tolist()
print(df.loc[df[cols_with_missing].isnull().any(axis=1), cols_with_missing].to_string())

In [ ]:
basement_cols = ['BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1']
#df[df[basement_cols].isnull().any(axis=1)][basement_cols]
df.loc[df[basement_cols].isnull().any(axis=1),basement_cols]

## Handle missing values


In [ ]:
basement_cols = ['BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1']
for col in basement_cols:
    df[col] = df[col].fillna('None')

In [ ]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'MasVnrType']
for col in none_cols:
    df[col] = df[col].fillna('None')

In [ ]:
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

In [ ]:
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [ ]:
df.isnull().sum().sum()

## EDA

In [ ]:
df.describe()

In [ ]:
df['SalePrice'].describe()

In [ ]:
# target variable's distribution
df['SalePrice'].hist(bins=50, figsize=(15,10))
plt.savefig('images/hist_saleprice.png', dpi=100)
plt.show()

In [ ]:
df['SalePrice'].plot(kind='box', figsize=(15,10), sharex=False)
plt.savefig('images/box_saleprice.png', dpi=100)
plt.show()

## Bivariate analysis + Univariate Analysis

### Analyze numeric features and their price correlations

In [ ]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])
correlations = numeric_df.corr()['SalePrice'].sort_values(ascending=False)
print(correlations)


In [ ]:
# numeric features histograms and boxplots

top_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
                 'TotalBsmtSF', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd',
                 'YearBuilt', 'YearRemodAdd']

for feature in top_features:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df[feature], bins=30)
    axes[0].set_title(f'{feature} - Histogram')

    axes[1].boxplot(df[feature])
    axes[1].set_title(f'{feature} - Boxplot')

    plt.tight_layout()
    plt.savefig('images/histograms&boxplots_top_numeric_features.png', dpi=100)
    plt.show()

### Analyze categorical features and their price correlations

In [ ]:
# ranks every categorical column by how much it separates cheap vs. expensive houses
categorical_cols = df.select_dtypes(include=['object']).columns

price_range_by_category = {}
for col in categorical_cols:
  group_means = df.groupby(col)['SalePrice'].mean()
  price_range_by_category[col] = group_means.max() - group_means.min()

price_range_series = pd.Series(price_range_by_category).sort_values(ascending=False)
print(price_range_series)



In [ ]:
# which categorical features high ranking is trustworthy
print(df.groupby('Utilities')['SalePrice'].agg(['mean', 'count']))
print(df.groupby('PoolQC')['SalePrice'].agg(['mean', 'count']))
print(df.groupby('ExterQual')['SalePrice'].agg(['mean', 'count']))
print(df.groupby('Neighborhood')['SalePrice'].agg(['mean', 'count']))
print(df.groupby('KitchenQual')['SalePrice'].agg(['mean', 'count']))
print(df.groupby('BsmtQual')['SalePrice'].agg(['mean', 'count']))

In [ ]:
print("Selected Categorical Features are:'ExterQual', 'KitchenQual', 'BsmtQual', 'Neighborhood' ")

In [ ]:
# box plot for bivariate analysis for categorical features
categorical_features = ['ExterQual', 'KitchenQual', 'BsmtQual', 'Neighborhood']

for feature in categorical_features:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=feature, y='SalePrice', data=df)
    plt.xticks(rotation=90)
    plt.title(f'{feature} vs SalePrice')
    plt.show()

In [ ]:
# scatter plot for bivariate analysis for numeric features
top_numeric = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
                'TotalBsmtSF', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd']

for feature in top_numeric:
    plt.figure(figsize=(6,4))
    plt.scatter(df[feature], df['SalePrice'], alpha=0.4)
    plt.xlabel(feature)
    plt.ylabel('SalePrice')
    plt.title(f'{feature} vs SalePrice')
    plt.show()

## Data Cleaning and Preprocessing

In [ ]:
df[(df['GrLivArea'] > 4000)][['GrLivArea', '1stFlrSF', 'TotalBsmtSF', 'TotRmsAbvGrd', 'SalePrice']]

In [ ]:
# remove anomilies - removal rule
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))]

In [ ]:
df.shape

In [ ]:
## Log Transformation of SalePrice (to reduce right skew)
df['SalePrice_log'] = np.log(df['SalePrice'])
df['SalePrice_log'].hist(bins=30)

### #categorical encoding strategy for data preprocessing---(Hot Encoding)

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
print(categorical_cols)

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
print(df.shape)
print(df_encoded.shape)